# CO543: Image Processing - Lab 7

### E/21/245
### Madhushan S.K.A.K.

## Task 1 — Use Pretrained CNN as Fixed Feature Extractor

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights

In [2]:
# Device configuraions
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda:0


In [4]:
#Setup the dataset (CIFAR-10)
# changing our images as common as ResNet18 train
transform = transforms.Compose([
    transforms.Resize((224, 224)), # ResNet wants 224,224 size images only
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # ImageNet standards
])

In [5]:
# Training dataset
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)

100%|██████████| 170M/170M [00:03<00:00, 48.0MB/s]


In [6]:
trainloader = torch.utils.data.DataLoader(trainset, batch_size=32, shuffle=True, num_workers=2)

In [7]:
# Validation dataset
valset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

In [8]:
valloader = torch.utils.data.DataLoader(valset, batch_size=32, shuffle=False, num_workers=2)

In [9]:
# Model Setup (Transfer Learning)
print("Setting up ResNet18...")
model = resnet18(weights=ResNet18_Weights.DEFAULT)

Setting up ResNet18...
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 188MB/s]


In [10]:
# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

In [11]:
# Replace the final layer
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 10) # CIFAR-10 has 10 classes
model = model.to(device)

In [12]:
# Loss Function & Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

In [13]:
epochs = 3
print("Starting training...")
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for i, (inputs, labels) in enumerate(trainloader):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad() # remove old gradients
        outputs = model(inputs)
        loss = criterion(outputs, labels) # finding the loss
        loss.backward() # Backpropagation
        optimizer.step()

        running_loss += loss.item()

        # showing the progress after each 500 batches
        if (i+1) % 500 == 0:
            print(f"Epoch [{epoch+1}/{epochs}], Step [{i+1}/{len(trainloader)}], Loss: {loss.item():.4f}")

    print(f"--> Epoch {epoch+1} Average Training Loss: {running_loss/len(trainloader):.4f}")

Starting training...
Epoch [1/3], Step [500/1563], Loss: 0.7911
Epoch [1/3], Step [1000/1563], Loss: 0.7548
Epoch [1/3], Step [1500/1563], Loss: 0.4696
--> Epoch 1 Average Training Loss: 0.8021
Epoch [2/3], Step [500/1563], Loss: 0.5848
Epoch [2/3], Step [1000/1563], Loss: 0.7878
Epoch [2/3], Step [1500/1563], Loss: 0.8515
--> Epoch 2 Average Training Loss: 0.6413
Epoch [3/3], Step [500/1563], Loss: 0.7063
Epoch [3/3], Step [1000/1563], Loss: 0.8219
Epoch [3/3], Step [1500/1563], Loss: 0.3223
--> Epoch 3 Average Training Loss: 0.6230


In [15]:
# Validation Accuracy
print("Evaluating on Validation Data...")
model.eval()
correct = 0
total = 0
with torch.no_grad(): # without gradients
    for inputs, labels in valloader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"\n Task 1 Completed! Validation Accuracy: {100 * correct / total:.2f}%")

Evaluating on Validation Data...

 Task 1 Completed! Validation Accuracy: 78.52%


**Output Observation:-**

- Training Loss: Decreased steadily from 0.8021 to 0.6230 over 3 epochs.

- Validation Accuracy: Reached 78.52%.

- Meaning:- The model learned to classify the new images quite well by only updating the very last layer. The pre-trained weights in the earlier layers successfully acted as a strong foundation

**More to think:- If your dataset is simple and small, do you think this setup is enough?**

- Yes, it is enough.
- Reason 1 (Prevents Overfitting):-  A small dataset does not have enough examples to train millions of parameters from scratch. If we try to train everything, the model will just memorize the training images and fail on new ones (overfitting).

 - Reason 2 (Generic Features are Sufficient):-  Early layers of the ResNet model already know how to find basic edges, colors, and simple shapes. For a simple dataset, these basic features combined with a newly trained final layer are all we need.

## Task 2 — Fine-Tune the Last Block

In [16]:
# Unfreeze the last convolutional block
print("Unfreezing the last block (layer4)...")
for param in model.layer4.parameters():
    param.requires_grad = True

Unfreezing the last block (layer4)...


In [17]:
# createing the new optimizer
smaller_lr = 1e-4 # use smalllerning rate


In [18]:
optimizer_finetune = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=smaller_lr)
criterion = nn.CrossEntropyLoss()

In [19]:
# Fine-Tuning Training Loop
epochs_finetune = 3 # for now just do for 3 epoches only
print("Starting Fine-Tuning...")
for epoch in range(epochs_finetune):
    model.train()
    running_loss = 0.0
    for i, (inputs, labels) in enumerate(trainloader):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer_finetune.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_finetune.step()

        running_loss += loss.item()

        if (i+1) % 500 == 0:
            print(f"Fine-Tune Epoch [{epoch+1}/{epochs_finetune}], Step [{i+1}/{len(trainloader)}], Loss: {loss.item():.4f}")

    print(f"> Fine-Tune Epoch {epoch+1} Average Training Loss: {running_loss/len(trainloader):.4f}")

Starting Fine-Tuning...
Fine-Tune Epoch [1/3], Step [500/1563], Loss: 0.4428
Fine-Tune Epoch [1/3], Step [1000/1563], Loss: 0.2153
Fine-Tune Epoch [1/3], Step [1500/1563], Loss: 0.1188
> Fine-Tune Epoch 1 Average Training Loss: 0.4080
Fine-Tune Epoch [2/3], Step [500/1563], Loss: 0.0554
Fine-Tune Epoch [2/3], Step [1000/1563], Loss: 0.0983
Fine-Tune Epoch [2/3], Step [1500/1563], Loss: 0.2057
> Fine-Tune Epoch 2 Average Training Loss: 0.1579
Fine-Tune Epoch [3/3], Step [500/1563], Loss: 0.1332
Fine-Tune Epoch [3/3], Step [1000/1563], Loss: 0.0385
Fine-Tune Epoch [3/3], Step [1500/1563], Loss: 0.1663
> Fine-Tune Epoch 3 Average Training Loss: 0.0831


In [20]:
# Task 2 New Validation Accuracy
print("Evaluating Fine-Tuned Model...")
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in valloader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"\n Task 2 Completed! New Validation Accuracy: {100 * correct / total:.2f}%")

Evaluating Fine-Tuned Model...

 Task 2 Completed! New Validation Accuracy: 90.12%


**Output Observation:-**

- Training Loss: Dropped very fast and reached a very low value (from 0.4080 down to 0.0831). This shows a faster convergence rate.

- Validation Accuracy: Jumped to 90.12%.

- Meaning: By unfreezing the last convolutional block, the model adjusted its complex feature detection to specifically match our new dataset (CIFAR-10), leading to a massive 11.6% increase in accuracy.

- Note on Overfitting: The training loss is very low (0.0831). If we trained for more epochs, the model might start overfitting. We stopped at a good time.

**More to think: Why adjust only higher layers instead of all layers at once?**

- Reason 1 (Universal vs. Specific Features):- Early layers learn universal features like straight lines, curves, and dots. These are the same in almost every image in the world, so we do not need to change them. Higher layers learn complex, specific shapes (like a dog's ear or a car's wheel). We only adjust the higher layers to adapt to our specific dataset.

- Reason 2 (Stability):- If we adjust all layers at once, we might destroy the valuable basic knowledge the model already learned from ImageNet.

- Reason 3 (Hardware Efficiency):- Training only the higher layers saves GPU memory and computing time.

## Task 3 — Compare & Interpret

**Comparison: Fixed-Feature vs. Partial Fine-Tuning**

- Accuracy:- Partial fine-tuning (Task 2) achieved a much higher validation accuracy (90.12%) compared to the fixed-feature extractor (78.52%).

- Stability:- The fixed-feature method is highly stable because only the final classifier is learning. Partial fine-tuning is more sensitive; it required a smaller learning rate ($10^{-4}$) to remain stable and prevent destroying the pre-trained weights.

- Training Behavior:- In Task 1, the loss decreased gradually. In Task 2, the training loss dropped very quickly to a low value (0.0831), showing a faster convergence rate, but also indicating that it could overfit if trained for too many epochs.

**When to use Freezing vs Fine-Tuning**

- Freezing works well when:- The new dataset is very small, and the images are similar to the original training data, The basic shapes and colors already learned are enough.

- Fine-tuning is necessary when:- The new dataset has completely different textures, shapes, or a "domain shift" (example:- medical X-rays, satellite images, or microscopic cells). It is also used when you have a large dataset that can safely update the model without overfitting.

**More to think": Why can fine-tuning on a very small dataset cause overfitting faster than using the frozen model?**

- Fine-tuning unfreezes complex layers, meaning the model suddenly has millions of adjustable parameters. If the dataset is very small, the model has too much "brainpower" for too little data. Instead of learning general rules, it simply memorizes the exact training images (like memorizing answers for a test).

## Final Reflection

- Transfer learning dramatically speeds up development by reusing weeks of GPU training and millions of images of learned features; instead of starting from zero, the model already understands fundamental visual concepts, allowing us to reach high accuracy in just a few minutes with minimal data. However, this approach can fail if there is a massive "domain gap" between the original data and the new data (for example, trying to use an ImageNet model trained on dogs and cars to analyze audio spectrograms or complex seismic charts). It can also fail if we use a high learning rate during fine-tuning, which causes "catastrophic forgetting," erasing all the valuable pre-trained knowledge.